# How Much of a Tree Is Luck?

**Notebook 1 of 5.** A decision tree is built entirely from its training rows. Change the
rows, and how different is the tree? The answer is why the rest of this repository exists.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

## 1. The data

`data/concrete.csv`: 1,030 laboratory recipes, which this repository calls **mixes**. Seven
ingredients in kilograms per cubic meter, the **age** in days at testing, and
`compressive_strength` in megapascals as the target.

Every notebook holds out the same 20% with `random_state=42`, so the scores compare.

In [ ]:
df = pd.read_csv("../data/concrete.csv")

X = df.drop(columns="compressive_strength")
y = df["compressive_strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"{len(X_train)} training mixes, {len(X_test)} test mixes")

## 2. One tree, for reference

In [ ]:
# no limits, so the tree splits until nearly every leaf holds a single mix
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

print(f"training RMSE: {root_mean_squared_error(y_train, tree.predict(X_train)):.2f} MPa")
print(f"test RMSE:     {root_mean_squared_error(y_test, tree.predict(X_test)):.2f} MPa")

**0.84 MPa on mixes it was given, 7.13 on mixes it has not seen.** That gap is overfitting,
and it is not the subject here. The question is whether 7.13 belongs to *the method* or to
*these 824 rows*.

## 3. The same algorithm, half the data each

Two halves of the training mixes, sharing no rows, one tree on each. Same code, same
`random_state`, same size. Only the rows differ.

In [ ]:
half_a, half_b, y_a, y_b = train_test_split(
    X_train, y_train, test_size=0.5, random_state=0
)

tree_a = DecisionTreeRegressor(random_state=42).fit(half_a, y_a)
tree_b = DecisionTreeRegressor(random_state=42).fit(half_b, y_b)

for name, fitted in [("tree A", tree_a), ("tree B", tree_b)]:
    # tree_.feature[0] and threshold[0] describe the root: the first question asked
    root = X.columns[fitted.tree_.feature[0]]
    print(f"{name}: root question {root} <= {fitted.tree_.threshold[0]:.1f}, "
          f"test RMSE {root_mean_squared_error(y_test, fitted.predict(X_test)):.2f} MPa")

Both open with `age <= 21.0` and both score about 8.5, which makes them look like minor
variations on each other. Compare what they predict.

In [ ]:
predictions_a = tree_a.predict(X_test)
predictions_b = tree_b.predict(X_test)

comparison = pd.DataFrame({
    "true": y_test.to_numpy()[:5],
    "tree A": predictions_a[:5],
    "tree B": predictions_b[:5],
}).round(1)
print(comparison.to_string(index=False))

gap = np.abs(predictions_a - predictions_b)
print(f"\nmean disagreement {gap.mean():.2f} MPa, largest {gap.max():.2f} MPa")

> **Exercise:**
>
> The second test mix really measured 55.9 MPa. What did each tree say, and how far apart are
> the two trees on a typical mix?

<details><summary>
Click here to compare your answer...
</summary>

**Tree A says 39.4, tree B says 54.1**, for the same recipe. Over all 206 test mixes they
differ by **7.57 MPa on average** and by **42.12** at worst — more than either tree's own
error. Two models from the same method and the same experiment disagree with each other more
than either disagrees with reality.

</details>

## 4. The words for this

- **Variance**: how far predictions move when the training rows change. Section 3 measured it.
- **Bias**: how far they sit from the truth on average, because the model is too simple.

A deep tree has **low bias and high variance**; a shallow one has the reverse. Measuring
variance properly needs many training sets, and there is only one dataset.

## 5. Making many training sets out of one

A **bootstrap sample** draws $n$ rows from $n$ **with replacement**: same size as the
original, some rows drawn twice or more, some not at all. Each one is a plausible version of
the experiment you might have run instead.

Ten draws from ten row numbers show the shape:

In [ ]:
rng = np.random.default_rng(42)

# ten draws from the positions 0 to 9, each draw free to repeat an earlier one
print(np.sort(rng.integers(0, 10, size=10)))

Position 0 is drawn three times; 1, 3, 5 and 9 are never drawn at all.

> **Exercise:**
>
> Draw a bootstrap sample of the positions of `X_train`, using a fresh
> `np.random.default_rng(42)`. What share of the training mixes appears in it at least once?

<details><summary>
Click here for a hint...
</summary>

`rng.integers(0, n, size=n)` with `n = len(X_train)`, then `np.unique` to count the distinct
positions.

</details>

In [ ]:
rng = np.random.default_rng(42)
n = len(X_train)

# n positions drawn with replacement: the same size as the training set
sample = rng.integers(0, n, size=n)
distinct = len(np.unique(sample))

print(f"distinct mixes: {distinct} of {n} ({distinct / n:.1%})")

<details><summary>
Click here to compare your answer...
</summary>

**523 of 824, 63.5%**, leaving 301 out.

Not an accident of this draw: all $n$ draws miss a given row with probability $(1 - 1/n)^n$,
which approaches $1/e \approx 0.368$. A bootstrap sample holds about **63.2%** of the rows,
whatever the dataset. The 37% left out are notebook 04's subject.

</details>

## 6. Twenty trees, twenty answers

One tree per bootstrap sample, and a row of test predictions from each.

In [ ]:
def bootstrap_predictions(max_depth=None, n_trees=20, seed=42):
    """Fit one tree per bootstrap sample of the training set and predict the test set.

    Args:
        max_depth: depth limit for every tree; None means no limit.
        n_trees: how many bootstrap samples, and so how many trees.
        seed: seed for the generator that draws the samples.

    Returns:
        array of shape (n_trees, number of test mixes), one row of predictions per tree.
    """
    rng = np.random.default_rng(seed)
    n = len(X_train)
    predictions = []
    for _ in range(n_trees):
        sample = rng.integers(0, n, size=n)

        # iloc, because a bootstrap sample is a list of positions, repeats included
        fitted = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
        fitted.fit(X_train.iloc[sample], y_train.iloc[sample])
        predictions.append(fitted.predict(X_test))
    return np.array(predictions)


deep = bootstrap_predictions()
print(deep.shape)

In [ ]:
mixes = np.arange(10)

plt.figure(figsize=(9, 4.5))

# one column of dots per test mix, one dot per tree, spread sideways to separate them
offsets = np.linspace(-0.3, 0.3, len(deep))
for mix in mixes:
    plt.scatter(mix + offsets, deep[:, mix], color="#252629", alpha=0.5, s=20)
plt.scatter(mixes, y_test.iloc[:10], color="#FF4A11", marker="D", s=70, zorder=3,
            label="true strength")

plt.xticks(mixes, [f"mix {m + 1}" for m in mixes])
plt.ylabel("compressive strength (MPa)")
plt.title("Twenty trees, one prediction each")
plt.legend()
plt.show()

> **Exercise:**
>
> Put a number on that scatter:
>
> 1. the test RMSE of each of the twenty trees, as a smallest, a largest and a mean;
> 2. the standard deviation of the twenty predictions for each mix, averaged over the mixes.
>    Call that the **spread**.

<details><summary>
Click here for a hint...
</summary>

Loop over the rows of `deep` for the RMSEs. `deep.std(axis=0)` runs down the trees and leaves
one standard deviation per test mix.

</details>

In [ ]:
# each row of deep is one tree's predictions, so one RMSE per row
rmses = [root_mean_squared_error(y_test, row) for row in deep]
print(f"RMSE per tree: {min(rmses):.2f} to {max(rmses):.2f}, mean {np.mean(rmses):.2f} MPa")

# axis=0 runs down the trees: one spread per test mix, then averaged
print(f"spread: {deep.std(axis=0).mean():.2f} MPa")

<details><summary>
Click here to compare your answer...
</summary>

**RMSE 7.06 to 9.05, mean 8.11 MPa. Spread 4.95 MPa.**

Every one is worse than section 2's 7.13, each having seen about 63% of the distinct mixes.
And a typical prediction moves by 5 MPa on nothing but which sample its tree was handed.

</details>

## Summary

- **Different rows give a different tree**: two trees from disjoint halves disagreed by
  7.57 MPa.
- **Variance** is that movement, **bias** is being wrong on average, and a deep tree has low
  bias and high variance.
- A **bootstrap sample** holds about **63%** of the rows.
- Twenty bootstrap trees are each **worse** than one tree on all the data, 8.11 against 7.13,
  and disagree by 4.95 MPa.

Twenty mediocre models that disagree sounds worse than one. Next, in `02_bagging.ipynb`: why
it is better.